# Amazon Product Query Assistant - EDA
Alan Liu, Zhihao Xie

In [6]:
import os
import requests
from pathlib import Path

In [12]:
data_folder = Path("../data/raw")
data_folder.mkdir(parents=True, exist_ok=True)

## Load Data

### An overview of the dataset

In [29]:
def count_num_of_records(file_path):
    number_of_records = 0
    with gzip.open(file_path, 'rt', encoding='utf-8') as f:
        for line in f:
            data = json.loads(line)
            # Asked Gemini: how to print JSON in format
            product_key = data.get("asin")
            if product_key:
                number_of_records += 1
    return number_of_records

In [ ]:
meta_path = data_folder / "meta_Sports_and_Outdoors.jsonl.gz"
print(f"Number of products: {count_num_of_records(meta_path)}")

### Inspection of sample records:
Looks like the title of the product is in `meta_Sports_and_Outdoors.jsonl.gz`

In [7]:
import gzip
import json
from collections import Counter

### Examine Meta Data
It looks like the following fields are useful for retrieval:
- title
- average_rating: average product rating.
- features: specification of products
- description: description of product
- price
- categories: keyword
- parent_asin

In [15]:
# Gemini's approach to reading .jsonl.gz without unzipping
with gzip.open(meta_path, 'rt', encoding='utf-8') as f:
    for line in f:
        data = json.loads(line)
        # Asked Gemini: how to print JSON in format
        print(data.keys())
        print(json.dumps(data, indent=4))
        print(data['videos'])
        print(data['store'])
        print(data['categories'])
        print(data['details'])
        break

dict_keys(['main_category', 'title', 'average_rating', 'rating_number', 'features', 'description', 'price', 'images', 'videos', 'store', 'categories', 'details', 'parent_asin', 'bought_together'])
{
    "main_category": null,
    "title": "Sure-Grip Zombie Wheels Low 59mm 4 Pack",
    "average_rating": 4.5,
    "rating_number": 84,
    "features": [
        "Pre-packaged in sets of 4",
        "Low profile 59mm x 38mm",
        "89a w/purple hub, 92a w/black hub, 95a w/red hub, 98a w/green hub",
        "Made in the U.S.A.",
        "Anodized Aluminum Hub"
    ],
    "description": [
        "All Zombie wheels are made in the USA. Zombie wheels feature anodized aluminum hubs for maximum durability and precise feel while maintaining rock solid stability. This allows our unique urethane compounds to deliver all your power to the floor. Choose the Zombie combination that fits your skating style and surface. Zombie Aluminum Core \u2013 Designed in house and manufactured using state of the 

### Examine Review Data
Tried to find one review of the above product. Relevant columns:
- title: review title
- text: review text

Will extract all the reviews associated with the selected `parent_asin`.

In [28]:
# Examine review data
data_folder = Path("../data/raw")
review_path = data_folder / "Sports_and_Outdoors.jsonl.gz"
# Gemini's approach to reading .jsonl.gz without unzipping
with gzip.open(review_path, 'rt', encoding='utf-8') as f:
    for line in f:
        data = json.loads(line)
        # Asked Gemini: how to print JSON in format
        if data.get("parent_asin") == 'B01HDXC8AG':
            print(json.dumps(data, indent=4))
            break

{
    "rating": 5.0,
    "title": "Excellent wheels",
    "text": "These replaced older wheels.  I mean from 80s old.  They are right height and solid.  The grip is a bit more than I want but that is no fault of wheels. I knew when I bought them they\u2019d be grippier but I didn\u2019t want to go less grippy.  I\u2019m very happy with performance.",
    "images": [],
    "asin": "B0157O33ES",
    "parent_asin": "B01HDXC8AG",
    "user_id": "AEHRHCKAPFO5RSX3VM73MZUXYJNA",
    "timestamp": 1520302057097,
    "helpful_vote": 4,
    "verified_purchase": true
}
